In [1]:
import numpy as np
import matplotlib as mpl
import glob
import os
import matplotlib.pyplot as plt
import numpy.ma as ma
from scipy import stats
import xarray as xr
import commondefs as myF
from mpl_toolkits.basemap import Basemap
import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.rcParams.update({'font.size': 10})

In [2]:
mydir = '../processed_netcdf/'

In [3]:
def is_feb(month):
    return month==2

def is_sep(month):
    return month==9

In [4]:
moddir = '../processed_netcdf/CMIP6_sic_1979-2018/'
allfiles = sorted([f for f in os.listdir(moddir) if 'siconc_SImon' in f])
modnames = [f.split('_')[2] for f in allfiles]

obsdir='../processed_netcdf/OBS_sic_1979-2018/'
obsnames = ['Bootstrap','NASA-Team','osisaf']
modnames = sorted(list(set(modnames)))
nobs = len(obsnames)
names = obsnames+ modnames
print(names)
nmod = len(names)
print(len(names))

['Bootstrap', 'NASA-Team', 'osisaf', 'ACCESS-CM2', 'ACCESS-ESM1-5', 'AWI-CM-1-1-MR', 'BCC-CSM2-MR', 'CAMS-CSM1-0', 'CESM2', 'CESM2-WACCM', 'CNRM-CM6-1', 'CNRM-ESM2-1', 'CanESM5', 'EC-Earth3', 'EC-Earth3-Veg', 'FGOALS-f3-L', 'FGOALS-g3', 'GFDL-CM4', 'GFDL-ESM4', 'HadGEM3-GC31-LL', 'INM-CM4-8', 'INM-CM5-0', 'IPSL-CM6A-LR', 'MIROC-ES2L', 'MIROC6', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NESM3', 'UKESM1-0-LL']
29


In [5]:
maps = [Basemap(projection='npstere', boundinglat=55, lon_0 = 0),Basemap(projection='spstere', boundinglat=-55, lon_0 = 180)]

/Users/lettieroach/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: MatplotlibDeprecationWarning: 
The dedent function was deprecated in Matplotlib 3.1 and will be removed in 3.3. Use inspect.cleandoc instead.
  """Entry point for launching an IPython kernel.


In [6]:
listds = []
lats = []
lons = []

for n,name in enumerate(names):
    
    if n<nobs:
        mydir = obsdir
    else:
        mydir = moddir
    ensfiles = [f for f in os.listdir(mydir) if name+'_' in f][0]
    ds = xr.open_dataset(mydir+ensfiles, chunks={'time': 12})
    if 'siconc' not in ds:
        if 'sic' in ds:
            ds = ds.rename({'sic':'siconc'})
        elif 'seaice' in ds:
            ds = ds.rename({'seaice':'siconc'})
        ds.siconc.values = 100.*ds.siconc.values
    #print(np.nanmax(ds.siconc.values))
    ds = myF.lat_renamer(ds)
    listds.append(ds)
    lat,lon = myF.get_lats(ds)
    lats.append(lat)
    lons.append(lon)

In [7]:
print(listds[5])

<xarray.Dataset>
Dimensions:    (bnds: 2, latitude: 192, longitude: 384, time: 480)
Coordinates:
  * time       (time) datetime64[ns] 1979-01-16T13:26:39 ... 2018-12-16T13:26:40
  * longitude  (longitude) float64 0.0 0.9375 1.875 2.812 ... 357.2 358.1 359.1
  * latitude   (latitude) float64 89.28 88.36 87.42 ... -87.42 -88.36 -89.28
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) datetime64[ns] dask.array<chunksize=(12, 2), meta=np.ndarray>
    siconc     (time, latitude, longitude) float32 99.7353 99.742966 ... 0.0 0.0
Attributes:
    CDI:               Climate Data Interface version 1.9.6 (http://mpimet.mp...
    Conventions:       CF-1.4
    history:           Fri Dec 06 16:07:30 2019: cdo selyear,1979/2018 AWICM_...
    source:            ECHAM6.3
    institution:       Max-Planck-Institute for Meteorology
    title:             hist1
    echam_version:     6.3.04
    advection:         Lin & Rood
    physics:           Modified ECMWF physics
    ra

In [ ]:
%%time
names[2] = 'OSISAF'
years = np.arange(1979,2019,1)
h = 1
for i,month in enumerate([1,8]):
    fig = plt.figure(figsize=(10,12))

    for mod in range(nmod):
        print(names[mod])
        
        if month==1:
            mydata = listds[mod].sel(time=is_feb(listds[mod]['time.month'])).siconc
        elif month==8:
            mydata = listds[mod].sel(time=is_sep(listds[mod]['time.month'])).siconc
        print(mydata.time.values)
        
        ni = len(lats[mod][:,0])
        nj = len(lons[mod][0,:])
        
        
        trend_mod = np.zeros([ni,nj])
        
        for i in range(int(ni/2)):
            for j in range(nj):
                ydat = mydata.values[:,i,j]
                if not np.all(ydat==0):
                    if (len(ydat[~np.isnan(ydat)])):
                        trend_mod[i,j], _, _, p_value, _ = stats.linregress(years[~np.isnan(ydat)], ydat[~np.isnan(ydat)])
                        #if p_value>0.05:
                        #    trend_mod[i,j] = 0.

        ax = plt.subplot(6,5,mod+1)
        ax.set_title(alphabet[mod]+' '+names[mod],fontsize=12,loc='left')
        m = maps[h]
        x, y = m(lons[mod],lats[mod])

        #mean sea ice contours
        mydata = listds[mod].groupby('time.month').mean(dim='time').siconc[month,:,:].values
        mydata = ma.masked_where(lats[mod]>0,mydata)
        mydata = ma.masked_invalid(mydata)
        mydata = ma.masked_where(mydata<0.1,mydata)
        CS1 = m.contour(x,y,mydata,levels=[15.,80.],linestyles=['--','-'],colors=['black','black','black'])   

        #trends in pcolor
        mydata = trend_mod
        mydata = ma.masked_where(lats[mod]>0,mydata)
        mydata = ma.masked_invalid(mydata)
        print(np.max(mydata*10.),np.min(mydata*10.))
        CS2 = m.pcolor(x,y,10.*mydata,cmap = plt.cm.seismic_r,vmin=-40.,vmax=40.)       
        m.drawcoastlines(color='black')
        m.fillcontinents(color='lightgrey')
        
        
        

    cbar_ax = fig.add_axes([0.78, 0.05, 0.18, 0.025]) #[left, bottom, width, height]
    cbar = fig.colorbar(CS2, cax=cbar_ax,  orientation='horizontal')
    cbar.ax.set_title('Sea ice concentration \n trend (%/dec)',fontsize=12)
    plt.tight_layout()
    #fig.savefig(savedir+'cmip6_mean_sic_sh_trend_'+str(month),bbox_inches='tight',dpi=600)
    plt.show()
    plt.close()

Bootstrap
['1979-02-01T00:00:00.000000000' '1980-02-01T00:00:00.000000000'
 '1981-02-01T00:00:00.000000000' '1982-02-01T00:00:00.000000000'
 '1983-02-01T00:00:00.000000000' '1984-02-01T00:00:00.000000000'
 '1985-02-01T00:00:00.000000000' '1986-02-01T00:00:00.000000000'
 '1987-02-01T00:00:00.000000000' '1988-02-01T00:00:00.000000000'
 '1989-02-01T00:00:00.000000000' '1990-02-01T00:00:00.000000000'
 '1991-02-01T00:00:00.000000000' '1992-02-01T00:00:00.000000000'
 '1993-02-01T00:00:00.000000000' '1994-02-01T00:00:00.000000000'
 '1995-02-01T00:00:00.000000000' '1996-02-01T00:00:00.000000000'
 '1997-02-01T00:00:00.000000000' '1998-02-01T00:00:00.000000000'
 '1999-02-01T00:00:00.000000000' '2000-02-01T00:00:00.000000000'
 '2001-02-01T00:00:00.000000000' '2002-02-01T00:00:00.000000000'
 '2003-02-01T00:00:00.000000000' '2004-02-01T00:00:00.000000000'
 '2005-02-01T00:00:00.000000000' '2006-02-01T00:00:00.000000000'
 '2007-02-01T00:00:00.000000000' '2008-02-01T00:00:00.000000000'
 '2009-02-01T00